# 第二题：实现一个高斯朴素贝叶斯分类器

实验内容：
1. 实现高斯朴素贝叶斯分类器
2. 计算模型的查准率，查全率，F1值

我们要实现一个可以处理连续特征的，服从高斯分布的朴素贝叶斯分类器。

## 符号

给定训练集 $T$

$$T = \{(x_1, y_1), (x_2, y_2), ···, (x_N, y_N)\}$$

其中，$x$ 为样本的特征，$y$ 是该样本对应的标记，下标表示对应的是第几个样本，上标表示第几个特征。训练集 $T$ 内一共 $\vert T \vert = N$ 个样本。

假设我们的任务是处理 $K$ 类分类任务，记类标记分别为 $c_1, c_2, ..., c_k$ 。

## 目标

我们的目标是对样本进行分类，这里我们用概率的方法，求 $P(Y = c_k \mid X = x), \ k = 1, 2, ..., K$ 中最大的那个概率对应的 $k$ 是哪个，也就是，给定样本 $x$ ，模型认为它是哪个类别的概率最大。

## 原理

由贝叶斯公式：

$$
\begin{aligned}
    P(Y = c_k \mid X = x) &= \frac{P(Y = c_k, X = x)}{P(X = x)} \\
                  &= \frac{P(X = x \mid Y = c_k)P(Y = c_k)}{\sum_kP(X = x \mid Y = c_k)P(Y = c_k)} \\
\end{aligned}
$$

这里，我们要求 $K$ 个概率中最大的那个，而这 $K$ 个概率的分母都相同，我们可以忽略分母部分，比较分子部分的大小，也就是比较 **先验概率** $P(Y = c_k)$ 和 **似然** $P(X = x \mid Y = c_k)$ 的乘积。

通过先验概率分布

$$
P(Y = c_k), \ k = 1, 2, ..., K
$$

和条件概率分布

$$
P(X = x \mid Y = c_k) = P(X^{(1)} = x^{(1)}, ···, X^{(n)} = x^{(n)} \mid Y = c_k), \ k = 1, 2, ..., K
$$

我们就可以得到联合概率分布 $P(X = x, Y = c_k)$ 。

**那么，问题就转化为了，如何求先验概率和似然？**

### 1. 先验概率 $P(Y = c_k)$ ：

先验概率的求解很简单，只要统计训练集中类别 $k$ 出现的概率即可。

$$
P(Y = c_k) = \frac{\mathrm{number} \ \mathrm{of}\ c_k}{N}
$$

### 2. 似然 $P(X=x\mid Y=c_k)$

朴素贝叶斯假设：**给定类别后，各特征条件独立**，因而

$$p(x\mid c_k)=\prod_{j=1}^{n}p(x^{(j)}\mid c_k).$$

这使联合似然转化为各个特征的条件似然之积。
连续变量使用概率密度，单点概率本身为零；以下高斯公式均指概率密度。


### 3. 连续特征的高斯条件密度

假设第 $j$ 个特征在类别 $c_k$ 下服从高斯分布：

$$p(x^{(j)}\mid c_k)=\frac{1}{\sqrt{2\pi\sigma_{k,j}^2}}
\exp\left[-\frac{(x^{(j)}-\mu_{k,j})^2}{2\sigma_{k,j}^2}\right].$$

均值和方差只从对应类别的训练样本估计：

$$\mu_{k,j}=\frac{1}{N_k}\sum_{i:y_i=c_k}x_i^{(j)},\qquad
\sigma_{k,j}^2=\frac{1}{N_k}\sum_{i:y_i=c_k}(x_i^{(j)}-\mu_{k,j})^2.$$

这里使用最大似然方差（`np.var` 的 `ddof=0`）。每个特征需要估计 $K$ 个均值和 $K$ 个方差。


## 综上

朴素贝叶斯分类器可以表示为：

$$
y = \mathop{\arg\max}_{c_k} P(Y = c_k) \prod_j P(X^{(j)} = x^{(j)} \mid Y = c_k)
$$

## 实现时的数值处理

多个很小的密度连乘容易下溢。由于对数是严格单调函数，最大化乘积等价于最大化对数之和：

$$\hat y=\arg\max_{c_k}\left\{\log P(c_k)
+\sum_{j=1}^{n}\log p(x^{(j)}\mid c_k)\right\}.$$

高斯密度的对数为

$$\log p(x^{(j)}\mid c_k)=-\frac12\log(2\pi\sigma_{k,j}^2)
-\frac{(x^{(j)}-\mu_{k,j})^2}{2\sigma_{k,j}^2}.$$

因此只需比较各类别的对数联合得分：

$$s_k(x)=\log P(c_k)-\frac12\sum_j
\left[\log(2\pi\sigma_{k,j}^2)+\frac{(x^{(j)}-\mu_{k,j})^2}{\sigma_{k,j}^2}\right].$$

分母 $p(x)$ 对所有类别相同，可以省略。得分不是归一化后的后验概率。
计算中在方差上加入很小的平滑量，以防止方差为零时出现除零或对零取对数。


# 1. 导入数据集

In [1]:
import numpy as np

In [2]:
from pathlib import Path

# 兼容从项目根目录或 solution 目录启动内核。
# 数据只从 data 目录读取，训练过程中不修改原始文件。
data_path = next(
    (base / "data" / "spambase" / "spambase.data"
     for base in (Path.cwd(), Path.cwd().parent)
     if (base / "data" / "spambase" / "spambase.data").is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("请从项目根目录或 solution 目录运行本 Notebook。")
spambase = np.loadtxt(data_path, delimiter=",")
spamx = spambase[:, :57]
spamy = spambase[:, 57].astype(int)
assert spamx.shape == (4601, 57)
assert set(np.unique(spamy)) == {0, 1}
assert np.isfinite(spamx).all() and (spamx >= 0).all()
print(f"Spambase: {spamx.shape[0]} 个样本，{spamx.shape[1]} 个特征")
print(f"正常邮件: {(spamy == 0).sum()}；垃圾邮件: {(spamy == 1).sum()}")


Spambase: 4601 个样本，57 个特征
正常邮件: 2788；垃圾邮件: 1813


# 2. 划分数据集

In [3]:
from sklearn.model_selection import train_test_split

# 保留模板指定的 60% 训练 / 40% 测试、随机种子 32。
trainX, testX, trainY, testY = train_test_split(
    spamx, spamy, test_size=0.4, random_state=32
)


In [4]:
trainX.shape, trainY.shape, testX.shape, testY.shape

((2760, 57), (2760,), (1841, 57), (1841,))

# 3. 实现高斯朴素贝叶斯

朴素贝叶斯的实现非常简单，但是首先需要大家掌握几个技巧的使用

### 1. python dict

python的字典，给定字典a，使用`a[key] = value`，将 `{key: value}` 键值对添加进a中

In [5]:
# 初始化字典，并添加键为 'a'、值为 1 的键值对。
test_dict = dict()
test_dict['a'] = 1


In [6]:
# 测试样例
print(test_dict['a']) # 1

1


### 2. np.mean

求均值，使用 `axis = 0` 这个参数对每列取均值，`axis = 1` 对每行取均值，使用 `keepdims = True` 使结果保持之前的维数。

In [7]:
# 取 spamy 为1 对应的 spamx 的行
test_matrix = spamx[spamy == 1, :]
print(test_matrix.shape)

(1813, 57)


In [8]:
# 沿样本轴求均值；keepdims=True 保留为 (1, 57)，便于后续广播。
test_mean = np.mean(test_matrix, axis=0, keepdims=True)


In [9]:
# 测试样例
print(test_mean.sum()) # 595.062044126
print(test_mean.shape) # (1, 57)

595.0620441257583
(1, 57)


### 3. np.var

求方差，使用 `axis = 0` 这个参数对每列取方差，`axis = 1` 对每行取方差，使用 `keepdims = True` 使结果保持之前的维数。

In [10]:
# 使用总体方差（ddof=0），对应高斯参数的最大似然估计。
test_var = np.var(test_matrix, axis=0, keepdims=True)


In [11]:
# 测试样例
print(test_var.sum()) # 772407.506004
print(test_var.shape) # (1, 57)

772407.5060043654
(1, 57)


### 4. 将test_mean和test_var存入字典test_dict中

将`{'mean': test_mean}` 和 `{'var': test_var}` 存入`test_dict`中

In [13]:
# 使用字符串键保存均值和方差数组。
test_dict['mean'] = test_mean
test_dict['var'] = test_var


In [14]:
# 测试样例
print(test_dict.keys())  # dict_keys(['a', 'mean', 'var'])

dict_keys(['a', 'mean', 'var'])


### 5. numpy的索引

我们在预测的时候，需要使用numpy索引的一个小技巧

给定一个列表，里面有3个字符串`'a', 'b', 'c'`，分别表示三个类别，给定一个`np.ndarray([1, 2, 0, 1, 0])`，我们可以执行以下代码观察numpy强大的索引功能

In [15]:
labels = ['a', 'b', 'c']

# 首先把 labels 变成 np.ndarray
np_labels = np.array(labels)

print(np_labels)

# 新建索引
index = np.array([1, 2, 0, 1, 0])

print(index)

# 使用index来检索np_labels
results = np_labels[index]

print(results)
print(type(results))

['a' 'b' 'c']
[1 2 0 1 0]
['b' 'c' 'a' 'b' 'a']
<class 'numpy.ndarray'>


可以看到，我们可以使用 `np.ndarray` 一次检索多个值，返回值会以 `np.ndarray` 的形式返回

### 6. np.argmax

这个是求最大值的下标用的，`axis`参数用来控制是每行还是每列

In [16]:
test_array = np.array([[0.9, 0.1],
                       [0.4, 0.6],
                       [0.1, 0.9]])

print(test_array)

[[0.9 0.1]
 [0.4 0.6]
 [0.1 0.9]]


In [17]:
np.argmax(test_array, axis = 0)

array([0, 2], dtype=int64)

可以看到，使用`axis = 0`，就是返回每列最大值的下标，分别是0和2。

In [18]:
# axis=1 沿每一行的列方向选择最大值，返回每个样本对应的类别列号。
test_argmax = np.argmax(test_array, axis=1)


In [19]:
# 测试样例
print(test_argmax) # [0 1 1]

[0 1 1]


接下来以类的形式实现高斯朴素贝叶斯。
训练时估计类别先验、每类各特征的均值和方差；预测时计算每个类别的对数联合得分。
朴素贝叶斯会在训练阶段学习参数，属于急切学习，而不是懒惰学习。


In [20]:
class myGaussianNB:
    """仅用 NumPy 实现连续特征的高斯朴素贝叶斯。"""

    def __init__(self):
        # 四个字典分别保存标签到列号的映射、先验概率、均值和方差。
        self.label_mapping = dict()
        self.probability_of_y = dict()
        self.mean = dict()
        self.var = dict()

    def _clear(self):
        """每次重新拟合时清除上一次训练的参数。"""
        self.label_mapping.clear()
        self.probability_of_y.clear()
        self.mean.clear()
        self.var.clear()

    def fit(self, trainX, trainY):
        """用训练集估计参数；trainX 为二维特征，trainY 为一维类别。"""
        trainX = np.asarray(trainX, dtype=float)
        trainY = np.asarray(trainY)
        if trainX.ndim != 2 or trainY.ndim != 1:
            raise ValueError("trainX 必须是二维数组，trainY 必须是一维数组。")
        if len(trainX) == 0 or len(trainX) != len(trainY) or trainX.shape[1] == 0:
            raise ValueError("训练集不能为空，且特征和标签的样本数必须一致。")
        if not np.isfinite(trainX).all():
            raise ValueError("特征不能包含 NaN 或无穷值。")

        self._clear()
        self.n_features_in_ = trainX.shape[1]
        labels = np.unique(trainY)
        self.label_mapping = {label: index for index, label in enumerate(labels)}
        # 只使用训练集方差；机器精度下界还可处理全部特征恒定的情形。
        self.epsilon_ = max(1e-9 * np.var(trainX, axis=0).max(), np.finfo(float).eps)

        for label in labels:
            x = trainX[trainY == label, :]
            # 先验 P(Y=label)：该类训练样本数 / 总训练样本数。
            self.probability_of_y[label] = x.shape[0] / trainX.shape[0]
            # 各特征的均值和最大似然方差；保留二维以便与测试矩阵广播。
            self.mean[label] = np.mean(x, axis=0, keepdims=True)
            self.var[label] = np.var(x, axis=0, keepdims=True)
            assert self.mean[label].shape == (1, trainX.shape[1])
            assert self.var[label].shape == (1, trainX.shape[1])
            self.var[label] += self.epsilon_
        return self

    def predict(self, testX):
        """计算各类对数联合得分，并返回得分最大的原始类别标记。"""
        if not self.label_mapping:
            raise ValueError("请先调用 fit 再预测。")
        testX = np.asarray(testX, dtype=float)
        if testX.ndim != 2 or testX.shape[1] != self.n_features_in_:
            raise ValueError("测试特征必须为二维，且特征数应与训练集相同。")
        if not np.isfinite(testX).all():
            raise ValueError("测试特征不能包含 NaN 或无穷值。")

        # 每行对应一个样本，每列对应一个类别；存储的是对数联合得分。
        results = np.empty((testX.shape[0], len(self.probability_of_y)))
        labels = [None] * len(self.probability_of_y)
        for label, index in self.label_mapping.items():
            py = self.probability_of_y[label]
            # 对每个特征计算高斯对数密度，再沿特征轴求和。
            sum_of_conditional_probability = np.sum(
                -0.5 * np.log(2.0 * np.pi * self.var[label])
                -0.5 * (testX - self.mean[label]) ** 2 / self.var[label],
                axis=1,
            )
            assert sum_of_conditional_probability.shape == (len(testX),)
            result = np.log(py) + sum_of_conditional_probability
            assert result.shape == (len(testX),)
            results[:, index] = result
            labels[index] = label

        # 取每个样本得分最大的类别列号，再转换回原始类别。
        np_labels = np.array(labels)
        max_prob_index = np.argmax(results, axis=1)
        assert max_prob_index.shape == (len(testX),)
        prediction = np_labels[max_prob_index]
        assert prediction.shape == (len(testX),)
        return prediction


In [21]:
from sklearn.metrics import accuracy_score

model = myGaussianNB()
model.fit(trainX, trainY)
prediction = model.predict(testX)
print(f"测试集准确率 accuracy = {accuracy_score(testY, prediction):.6f}")


测试集准确率 accuracy = 0.812602


# 4. 计算其他指标

采用模板指定的固定测试集，以垃圾邮件 `1` 为正类。
下表为实际执行结果，所有参数仅从训练集估计。

| 算法 | 准确率 accuracy | 查准率 precision | 查全率 recall | F1 |
| --- | --- | --- | --- | --- |
| myGaussianNB | 0.812602 | 0.682951 | 0.962079 | 0.798834 |


In [22]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB

test_metrics = {
    "accuracy": accuracy_score(testY, prediction),
    "precision": precision_score(testY, prediction, zero_division=0),
    "recall": recall_score(testY, prediction, zero_division=0),
    "f1": f1_score(testY, prediction, zero_division=0),
}
display(pd.DataFrame([test_metrics], index=["myGaussianNB"]).round(6))
display(pd.DataFrame(
    confusion_matrix(testY, prediction, labels=[0, 1]),
    index=["真实正常邮件", "真实垃圾邮件"],
    columns=["预测正常邮件", "预测垃圾邮件"],
))

# 用 sklearn 作为独立参照，核对手写实现的先验、均值、方差和预测。
reference = GaussianNB().fit(trainX, trainY)
np.testing.assert_array_equal(prediction, reference.predict(testX))
np.testing.assert_allclose(
    [model.probability_of_y[label] for label in reference.classes_],
    reference.class_prior_,
)
np.testing.assert_allclose(
    np.vstack([model.mean[label] for label in reference.classes_]), reference.theta_,
)
np.testing.assert_allclose(
    np.vstack([model.var[label] for label in reference.classes_]), reference.var_,
)
# 核对前面 NumPy 小练习是否满足题设。
assert test_dict['a'] == 1
assert test_mean.shape == test_var.shape == (1, 57)
np.testing.assert_allclose(test_mean.sum(), 595.062044126, rtol=1e-9)
np.testing.assert_allclose(test_var.sum(), 772407.506004, rtol=1e-9)
np.testing.assert_array_equal(test_argmax, [0, 1, 1])
print("验证通过：手写模型的参数及测试集预测与 sklearn GaussianNB 一致。")


,accuracy,precision,recall,f1
myGaussianNB,0.812602,0.682951,0.962079,0.798834


,预测正常邮件,预测垃圾邮件
真实正常邮件,811,318
真实垃圾邮件,27,685


验证通过：手写模型的参数及测试集预测与 sklearn GaussianNB 一致。


### 结果分析

固定测试集共 1841 个样本，准确率为 0.812602，查准率为 0.682951，查全率为 0.962079，F1 为 0.798834。正常邮件误判为垃圾邮件 318 封，垃圾邮件漏检 27 封。手写实现的先验、均值、平滑方差及全部测试预测均与 sklearn GaussianNB 一致，说明参数估计和对数得分公式实现相符。该结果采用一次固定训练/测试划分，与第一题十折均值的评价口径不同。

高斯模型直接对连续特征建模，无须二值化。Spambase 的词频、字符频率及大写字母统计
存在偏态、零值较多且特征之间相关，因此高斯分布和条件独立假设只是近似。
方差平滑解决数值稳定性问题；它不等同于对离散事件计数加一的拉普拉斯修正。
